In [ ]:
import importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import subprocess
import sys
import time

# Package Verification & Installation Setup
packages_to_check = [
    "optuna",
    "tqdm",
    "gdown",
    "catboost",
    "scikit-learn",
]

missing_packages = [
    pkg for pkg in packages_to_check
    if importlib.util.find_spec(pkg) is None
]

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info"
)

status = widgets.HTML(value="Starting setup...")
display(progress, status)

progress.value = 20
status.value = "Checking required packages..."

if missing_packages:
    step_val = 70 / len(missing_packages)

    for i, pkg in enumerate(missing_packages):
        status.value = f"Installing package: {pkg}..."

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True
        )

        progress.value = int(20 + (i + 1) * step_val)

progress.value = 100
progress.bar_style = "success"
status.value = "✓ All required packages are available."

# Imports
import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    cross_val_predict,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope


optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, recall_score)
import joblib


## 1. Load the data and hold out a test set

The group's EDA notebook (`eda.ipynb`) cleans the raw data and saves the result as `data/data_clean.csv`. The cleaning steps, and how many rows each one removed, are documented there.

> The `data/` folder is **not** in the repository. Download the dataset from the Drive link in the README and run the EDA notebook first, because that creates this file. The eda.ipynb also has the download built-in the code, so it can run, download the file and create the data folder conveniently.

We then split off **20% as a test set** and do not touch it again until the champion is chosen. Everything in between (cross-validation, grid search, Hyperopt) happens on the training 80% only.

| Set | Used for |
|---|---|
| **Train (80%)** | Cross-validation, model comparison, hyperparameter tuning |
| **Test (20%)** | One final check of the champion on data no model has ever seen |

`stratify=y` keeps the same class proportions in both halves.

> **Reproducibility:** every step that involves randomness uses `random_state=42`, so anyone who runs this notebook gets the same champion and the same model file.

In [ ]:
## 1.2 Settings
RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- The five OCEAN traits (column names start with their letter) ---------
TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# --- One fixed colour per personality type, used in every plot -------------
TARGET_COLORS = {
    "Resilient":        "#e07a5f",  # muted terracotta/coral
    "Overcontroller":   "#81b29a",  # sage/mint green
    "Undercontroller":  "#f2cc8f",  # warm ochre yellow
    "Moderate":         "#2b5c6f"   # petroleum blue (same as the primary theme colour)
}

target_order = list(TARGET_COLORS.keys())

# --- Colours for titles, info boxes and heatmaps -----------------------------
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

In [ ]:
# 1.3 Load data
DATA_FILE = Path("DATA") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

display(HTML(f"""
<table style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; 
              border-collapse:collapse; margin:8px 0; font-size:14px; text-align:left;">
    <tr>
        <td style="padding:10px 14px; border:none;" colspan="2"><b>Total:</b> {len(df):,} rows</td>
    </tr>
    <tr>
        <td style="padding:4px 14px; border:none;"><b>Train:</b> {len(X_train):,} rows</td>
        <td style="padding:4px 14px; border:none;"><b>Test:</b> {len(X_test):,} rows</td>
    </tr>
    <tr>
        <td style="padding:4px 14px 10px 14px; border:none;"><b>Features:</b> {X.shape[1]}</td>
        <td style="padding:4px 14px 10px 14px; border:none;"><b>Items:</b> {len(item_cols)}</td>
    </tr>
</table>
"""))

df.head()

In [ ]:
counts = y.value_counts()

target_overview = pd.DataFrame({"count": counts,"share_%": (counts / len(y) * 100).round(1)})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=target_order, hue=y, palette=TARGET_COLORS, legend=False)
plt.title("How many people of each personality type?", color=THEME["primary"])
plt.xlabel("")
plt.ylabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different
treatment:

- **numbers** (the 19 items + `age`): fill any gaps with the median, then scale
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode

A `ColumnTransformer` made of two small pipelines describes this once. Every model below
reuses it, so each model sees exactly the same prepared data and the comparison is fair.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app —
no label encoder needed.

### What the preprocessor actually does

`gender` and `hand` turn into 0/1 columns, and no gaps are left behind. We fit it on the
**training data only** here, just to look at it — inside the pipelines below it is refit
within every fold, so nothing leaks from a validation fold into the scaling.

In [ ]:
# 2. Column definitions
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

# 2.5 Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

display(preprocessor)

# 3 Pipeline Setup

In [ ]:
# 3.1. Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# 3.2. Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# 3.3. Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# 3.4a. Pipeline for Hist Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.4b. Pipeline for Hist Gradient Boosting (Tuned variant)
pipe_hgb_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.5. Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(random_state=42, verbose=0)),
    ]
)

df_pipelines = pd.DataFrame([
    {
        "Pipeline": "<code>pipe_logreg</code>",
        "Model": "LogisticRegression",
        "Family": "Linear Model",
        "Why?": "Simple, highly interpretable baseline."
    },
    {
        "Pipeline": "<code>pipe_knn</code>",
        "Model": "KNeighborsClassifier",
        "Family": "Distance-based",
        "Why?": "Classifies based on similar answers; scaling is important."
    },
    {
        "Pipeline": "<code>pipe_rf</code>",
        "Model": "RandomForestClassifier",
        "Family": "Bagging / Decision Trees",
        "Why?": "Captures non-linear relationships and feature interactions."
    },
    {
        "Pipeline": "<code>pipe_hgb</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Histogram-based boosting as a fast candidate for tabular data."
    },
    {
        "Pipeline": "<code>pipe_hgb_tuned</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Optimized variant of HistGradientBoosting for performance gains."
    },
    {
        "Pipeline": "<code>pipe_cat</code>",
        "Model": "CatBoostClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Robust boosting method, strong performance on structured data."
    }
])

html_table = f"""
<table style="width:100%; border-collapse:collapse; margin:12px 0; font-size:14px; font-family:sans-serif; text-align:left;">
    <thead>
        <tr style="background:{THEME['success_bg']}; border-bottom:2px solid {THEME['success_border']};">
            <th style="padding:10px 14px; color:{THEME['primary']};">Pipeline</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Model</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Family</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Why?</th>
        </tr>
    </thead>
    <tbody>
"""

for i, row in df_pipelines.iterrows():
    bg_color = "#fcfdfd" if i % 2 == 0 else f"{THEME['success_bg']}55"
    html_table += f"""
        <tr style="background:{bg_color}; border-bottom:1px solid #eaeaea;">
            <td style="padding:10px 14px;">{row['Pipeline']}</td>
            <td style="padding:10px 14px;">{row['Model']}</td>
            <td style="padding:10px 14px;">{row['Family']}</td>
            <td style="padding:10px 14px;">{row['Why?']}</td>
        </tr>
    """

html_table += """
    </tbody>
</table>
"""

display(HTML(html_table))

display(pipe_hgb)   # show one pipeline as an example: the shared preprocessor + the classifier

# 4 Base pipeline

In [ ]:
base_pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "Gradient Boosting": pipe_hgb,
    "CatBoost": pipe_cat
}
base_results = []
base_results_html = ""

# Track execution times for relative progress bar calculation
durations = []
temp_runs = []
for name, pipe in tqdm(base_pipelines.items(), desc="Training Base Models"):
    start_time = time.time()
    pipe.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    fold_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    test_score = pipe.score(X_test, y_test)
    
    durations.append(elapsed_time)
    temp_runs.append({
        "name": name,
        "model": pipe,
        "test_score": test_score,
        "cv_score": mean_cv,
        "elapsed_time": elapsed_time,
        "fold_scores": fold_scores,
        "std_cv": std_cv
    })

max_time = max(durations) if durations else 1

for run in temp_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["cv_score"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    fold_scores = run["fold_scores"]
    
    # Dynamic coloring based on execution time using THEME values
    time_color = THEME['success_border'] if elapsed_time < 3 else (THEME['success_border'] if elapsed_time < 10 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_time) * 100)

    # Compact fold display
    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    base_results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    base_results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Default)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>
        
        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        
        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

best_base_overall = max(base_results, key=lambda x: x["test_score"])

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Base Model Comparison (Without Tuning) & F1 Macro Details</h3>
    {base_results_html}
    
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Base Model: {best_base_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_base_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

## 5. Hyperparameter tuning — four models

CatBoost is excluded from hyperparameter tuning because the search would take substantially longer than for the other models. It is still included in the baseline comparison, but the tuning section focuses on Logistic Regression, K-Nearest Neighbors, Random Forest, and HistGradient Boosting.

A hyperparameter is a setting we choose before training (how deep a tree may grow, how strong
the regularisation is). Tuning means searching for the combination that scores best.

The brief asks for at least three models each with tuning, so we tune **all four**, with two
methods. Both score every candidate with the same cross-validation and the same macro F1 as
section 5, so every number in this notebook stays comparable:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt (TPE)** | Learns from the trials it already ran and samples where the score looks promising | Many settings, or continuous ranges |

Each tuned pipeline is also scored **once** on the held-out test set. We report that number
next to the CV score, but the champion is picked by the CV score only (see section 7).

> **Note:** The brief asks for tuning on at least three models, so we tune all four. Without that rule we might have skipped Random Forest and K-Nearest Neighbors, because they were furthest behind before tuning. The results below show that would have been a mistake for Random Forest: it gains the most of all four models from tuning (about +0.04 macro F1) and ends up close to Logistic Regression.

### 5.1 Grid search

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step called `classifier`"*. That is how we reach inside the pipeline. Since the whole pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks from the validation fold into the scaling.

In [ ]:
# 5.1 Dictionary for GridSearch Tuning & Models Configuration
models_config = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10],
            "classifier__solver": ["lbfgs"],
        },
    },
    "KNN": {
        "pipeline": pipe_knn,
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
        },
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
        },
    },
    "HistGradient Boosting": {
        "pipeline": pipe_hgb,
        "params": {
            "classifier__learning_rate": [0.01, 0.05, 0.1],
            "classifier__max_iter": [100, 200],
        },
    },
    "HistGradient Boosting Tuned": {
        "pipeline": pipe_hgb_tuned,
        "params": {
            "classifier__max_leaf_nodes": [31, 32],
            "classifier__min_samples_leaf": [20, 60],
            "classifier__class_weight": ["balanced"],
        },
    },
    # Too slow! 
    # "CatBoost": {
    #     "pipeline": pipe_cat,
    #     "params": {
    #         "classifier__iterations": [400, 500, 600],
    #         "classifier__depth": [5, 6, 7],
    #         "classifier__learning_rate": [0.03, 0.05, 0.07],
    #         "classifier__l2_leaf_reg": [4, 6, 8]
    #     }
    # }
}

In [ ]:
results = []
results_html = ""

# Track execution times for relative progress bar calculation
durations_grid = []
temp_grid_runs = []

for name, config in tqdm(models_config.items(), desc="Grid Search Tuning"):
    start_time = time.time()
    
    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring={
            "f1_macro": "f1_macro",
            "accuracy": "accuracy"
        },
        refit="f1_macro",
        return_train_score=True,
        n_jobs=-1,
        error_score="raise"
    )
    
    grid_search.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    best_model = grid_search.best_estimator_
    best_index = grid_search.best_index_
    search_scores = grid_search.cv_results_
    
    mean_cv = search_scores["mean_test_f1_macro"][best_index]
    std_cv = search_scores["std_test_f1_macro"][best_index]
    test_score = best_model.score(X_test, y_test)
    
    durations_grid.append(elapsed_time)
    temp_grid_runs.append({
        "name": name,
        "model": best_model,
        "mean_cv": mean_cv,
        "std_cv": std_cv,
        "test_score": test_score,
        "elapsed_time": elapsed_time,
        "best_params": grid_search.best_params_
    })

max_grid_time = max(durations_grid) if durations_grid else 1

for run in temp_grid_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["mean_cv"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    best_params = run["best_params"]
    fold_scores = cross_val_score(run["model"], X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    
    # Dynamische Farbcodierung für Laufzeit über THEME-Werte
    time_color = THEME['success_border'] if elapsed_time < 15 else (THEME['success_border'] if elapsed_time < 45 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_grid_time) * 100)
    
    # Beste Parameter übersichtlich formatieren (Präfixe wie 'classifier__' entfernen)
    best_params_str = ", ".join([f"<b>{k.replace('classifier__', '')}</b>: {v}" for k, v in best_params.items()])

    results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    # Kompakte Folds-Darstellung im Theme-Design
    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Grid Search)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>
        
        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>
        
        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        
        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

best_overall = max(results, key=lambda x: x["test_score"])
best_pipeline = best_overall["model"]

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Model Comparison, Tuning & F1 Macro Details</h3>
    {results_html}
    
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Model: {best_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

### 6.2 Hyperopt

The same four models, but now with **continuous** ranges: `learning_rate` anywhere between
0.02 and 0.3, not just the two values the grid could afford. Hyperopt tries a setting, sees
the score, and uses that to decide what to try next — so it covers a much wider space in the
same number of fits.

Three things to know about the syntax:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1, 10 equally likely)
- `scope.int(...)` is needed because `hp.quniform` returns a float, and `n_estimators` must be a whole number
- Hyperopt **minimises**, so the objective returns `-score`

In [ ]:
# 6.2 Dictionary for Hyperopt Tuning & Models Configuration and Spaces 
hyperopt_configs = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "space": {
            "classifier__C": hp.loguniform("classifier__C", np.log(0.01), np.log(10))
        }
    },
    "KNN": {
        "pipeline": pipe_knn,
        "space": {
            "classifier__n_neighbors": scope.int(hp.quniform("classifier__n_neighbors", 3, 11, 2))
        }
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "space": {
            "classifier__n_estimators": scope.int(hp.quniform("classifier__n_estimators", 50, 300, 50)),
            "classifier__max_depth": scope.int(hp.quniform("classifier__max_depth", 5, 30, 5)),
            "classifier__min_samples_split": scope.int(hp.quniform("classifier__min_samples_split", 2, 10, 2))
        }
    },
    "HistGradient Boosting": {
        "pipeline": pipe_hgb,
        "space": {
            "classifier__learning_rate": hp.uniform("classifier__learning_rate", 0.02, 0.3),
            "classifier__max_iter": scope.int(hp.quniform("classifier__max_iter", 50, 250, 50))
        }
    }
}


In [ ]:
hyperopt_results = []
hyperopt_results_html = ""
durations_hyper = []
temp_hyper_runs = []

for name, config in tqdm(hyperopt_configs.items(), desc="Hyperopt Tuning"):
    start_time = time.time()
    
    pipeline = config["pipeline"]
    space = config["space"]
    
    def objective(params):
        pipeline.set_params(**params)
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}
    
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
        verbose=0
    )
    
    elapsed_time = time.time() - start_time
    
    # Beste Parameter formatieren/konvertieren
    best_params = {}
    for k, v in best.items():
        if "n_estimators" in k or "max_depth" in k or "min_samples_split" in k or "n_neighbors" in k or "max_iter" in k:
            best_params[k] = int(v)
        else:
            best_params[k] = v
            
    # Bestes Modell final konfigurieren und trainieren
    pipeline.set_params(**best_params)
    pipeline.fit(X_train, y_train)
    
    fold_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    test_score = pipeline.score(X_test, y_test)
    
    durations_hyper.append(elapsed_time)
    temp_hyper_runs.append({
        "name": name,
        "model": pipeline,
        "mean_cv": mean_cv,
        "std_cv": std_cv,
        "test_score": test_score,
        "elapsed_time": elapsed_time,
        "best_params": best_params,
        "fold_scores": fold_scores
    })

max_hyper_time = max(durations_hyper) if durations_hyper else 1

for run in temp_hyper_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["mean_cv"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    best_params = run["best_params"]
    fold_scores = run["fold_scores"]
    
    time_color = THEME['success_border'] if elapsed_time < 15 else (THEME['success_border'] if elapsed_time < 45 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_hyper_time) * 100)
    
    best_params_str = ", ".join([f"<b>{k.replace('classifier__', '')}</b>: {v:.4g}" if isinstance(v, float) else f"<b>{k.replace('classifier__', '')}</b>: {v}" for k, v in best_params.items()])

    hyperopt_results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    hyperopt_results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Hyperopt)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>
        
        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>
        
        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        
        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

best_hyper_overall = max(hyperopt_results, key=lambda x: x["test_score"])

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Hyperopt Tuning Results & Performance Details</h3>
    {hyperopt_results_html}
    
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Hyperopt Model: {best_hyper_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_hyper_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

## 7. Results — the comparison and the winner

The best tuned run of each model, shown as one card per model: the best settings the search found, how long it
took, the cross-validation score it was selected by, and its score on the held-out test set.

The **champion is the run with the highest cross-validated macro F1** — the metric we committed to in section 3. The test score is reported alongside it as an independent check, not as the selection criterion (choosing by the test score would spend the one honest number we have).

In [ ]:
# 7.1 Combine all runs and select the champion by CV macro F1

def result_rows(runs, tuning):
    return [
        {
            "model": run["name"],
            "tuning": tuning,
            "pipeline": run["model"],
            "cv_f1_macro": run["cv_score"],
            "test_f1_macro": f1_score(y_test, run["model"].predict(X_test), average="macro"),
            "test_accuracy": accuracy_score(y_test, run["model"].predict(X_test)),
            "params": run["model"].get_params(deep=True) if tuning == "none" else run.get("best_params", {}),
        }
        for run in runs
    ]

# Use the tuned-run lists directly so the summary stays correct when searches are changed.
grid_runs = [
    {**run, "cv_score": run["mean_cv"]}
    for run in temp_grid_runs
]
hyperopt_runs = [
    {**run, "cv_score": run["mean_cv"]}
    for run in temp_hyper_runs
]
all_runs = pd.DataFrame(
    result_rows(temp_runs, "none")
    + result_rows(grid_runs, "grid search")
    + result_rows(hyperopt_runs, "hyperopt")
)

champion_row = all_runs.loc[all_runs["cv_f1_macro"].idxmax()]
CHAMPION_NAME = champion_row["model"]
CHAMPION_TUNING = champion_row["tuning"]
CHAMPION_CV = champion_row["cv_f1_macro"]
CHAMPION_TEST = champion_row["test_f1_macro"]
champion_test_acc = champion_row["test_accuracy"]
champion_params = champion_row["params"]
champion_pipeline = clone(champion_row["pipeline"])
champion_pipeline.fit(X_train, y_train)

# A simple, class-frequency-aware reference point.
majority_class = y_train.mode().iat[0]
baseline_pred = np.full(shape=len(y_test), fill_value=majority_class, dtype=object)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")

def format_params(params):
    if not params:
        return "the default parameters"
    return ", ".join(
        f"{key.replace('classifier__', '')}={value}" for key, value in params.items()
    )

why_selected = (
    f"It achieved the highest mean cross-validated macro F1 ({CHAMPION_CV:.3f}) "
    f"across all evaluated runs. Its held-out test macro F1 was {CHAMPION_TEST:.3f}."
)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; background:{THEME['success_bg']}; border-left:5px solid {THEME['success_border']}; border-radius:6px; padding:16px; margin:12px 0;">
  <div style="font-size:13px; color:#555; margin-bottom:5px;">Selected by cross-validated macro F1</div>
  <h2 style="margin:0 0 8px; color:{THEME['primary']};">Champion: {CHAMPION_NAME}</h2>
  <p style="margin:0 0 10px;">{why_selected}</p>
  <table style="border-collapse:collapse; width:100%; background:#fff;">
    <tr><th style="text-align:left; padding:7px;">Search method</th><td style="padding:7px;">{CHAMPION_TUNING.title()}</td></tr>
    <tr><th style="text-align:left; padding:7px;">CV macro F1</th><td style="padding:7px;"><b>{CHAMPION_CV:.3f}</b></td></tr>
    <tr><th style="text-align:left; padding:7px;">Test macro F1</th><td style="padding:7px;">{CHAMPION_TEST:.3f}</td></tr>
    <tr><th style="text-align:left; padding:7px;">Test accuracy</th><td style="padding:7px;">{champion_test_acc:.1%}</td></tr>
    <tr><th style="text-align:left; padding:7px;">Best parameters</th><td style="padding:7px;">{format_params(champion_params)}</td></tr>
  </table>
</div>
"""))


In [ ]:
### Confusion matrix: each row shows what share of that true class went where
# 7.2 Confusion matrix for the selected model
y_pred_test = champion_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_pred_test, labels=target_order, normalize="true")
plt.figure(figsize=(8, 5))
sns.heatmap(
    cm, annot=True, fmt=".0%", cmap=THEME["palette"],
    xticklabels=target_order, yticklabels=target_order, vmin=0, vmax=1,
    cbar_kws={"label": "Share of true class"}
)
plt.title(f"Confusion matrix: {CHAMPION_NAME} ({CHAMPION_TUNING})", color=THEME["primary"])
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

print("Rows are true classes. Values are normalized within each row, so the diagonal is recall.")


In [ ]:
# 7.3 Full leaderboard
leaderboard = all_runs[["model", "tuning", "cv_f1_macro", "test_f1_macro", "test_accuracy"]].copy()
leaderboard = leaderboard.sort_values(["cv_f1_macro", "test_f1_macro"], ascending=False).reset_index(drop=True)
leaderboard.index = leaderboard.index + 1
leaderboard.index.name = "rank"
leaderboard["tuning"] = leaderboard["tuning"].replace({"none": "untuned"})

display(
    leaderboard.style
    .format({"cv_f1_macro": "{:.3f}", "test_f1_macro": "{:.3f}", "test_accuracy": "{:.1%}"})
    .highlight_max(subset=["cv_f1_macro"], color=THEME["success_bg"])
    .set_caption("Runs are ranked by cross-validated macro F1; test results are shown only as a final check.")
)


## 8. The champion on the test set

The champion was chosen with cross-validation on the training data. Now we look at how it
behaves on the 20% it has never seen — per class, which is where the interesting part is.

In [ ]:
# 8. Detailed evaluation on the held-out test set
report = classification_report(
    y_test, y_pred_test, labels=target_order, target_names=target_order, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).T.loc[target_order + ["macro avg", "weighted avg"]]
display(report_df[["precision", "recall", "f1-score", "support"]].style.format({
    "precision": "{:.3f}", "recall": "{:.3f}", "f1-score": "{:.3f}", "support": "{:.0f}"
}))

print(f"Held-out test macro F1: {f1_score(y_test, y_pred_test, average='macro'):.3f}")
print(f"Held-out test accuracy: {accuracy_score(y_test, y_pred_test):.1%}")


## 9. Fit the champion on all the data and save it with joblib

Cross-validation on the training set showed which pipeline performs best, and the held-out test set confirmed that result.
Since the test score has already been measured, the test rows aren't needed for evaluation anymore. So for the final model we refit the champion pipeline on
the full dataset (training and test data combined), so it learns from every row we have before it goes into `app.py`.

We save the **complete pipeline** with `joblib`, not just the classifier. The pipeline contains
the preprocessing steps (imputation, scaling and one-hot encoding) and the tuned classifier.
When the app passes in a raw row of user answers, the pipeline transforms it exactly the same
way as the training data and then makes the prediction.

> The `models/` folder is listed in `.gitignore`, so it isn't committed to the repository.
> Running this notebook creates the model file again. Every step that involves randomness uses `random_state=42`, so you get the same model every time.

In [ ]:
# 9.1 Refit the selected pipeline on all available data and save it
final_pipeline = clone(champion_pipeline)
final_pipeline.fit(X, y)

model_path = Path("models") / "personality_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, model_path)
print(f"Saved the complete pipeline to: {model_path}")


## 11. Final decision

The closing paragraph the project brief asks for. The numbers are filled in from the results
above, so the text always matches the latest run, even if the cleaned data changes.

In [ ]:
# 11. Final decision
untuned = all_runs[all_runs["tuning"] == "none"].sort_values("cv_f1_macro", ascending=False)
matching_untuned = untuned[untuned["model"] == CHAMPION_NAME]
untuned_champion = matching_untuned.iloc[0] if not matching_untuned.empty else None
gain = CHAMPION_CV - untuned_champion["cv_f1_macro"] if untuned_champion is not None else None

model_rows = "\n".join(
    f"| **{row.model}** | **{row.cv_f1_macro:.3f}** |" if row.model == CHAMPION_NAME
    else f"| {row.model} | {row.cv_f1_macro:.3f} |"
    for row in untuned.itertuples()
)

final_decision = f"""
**Baseline comparison.** The default pipelines were evaluated with {N_SPLITS}-fold stratified cross-validation on the training set.

| Model | CV macro F1 (untuned) |
|---|---|
{model_rows}

**Final model.** {CHAMPION_NAME}, tuned with {CHAMPION_TUNING}, was selected because it achieved the highest cross-validated macro F1: **{CHAMPION_CV:.3f}**. Its macro F1 on the held-out test set was **{CHAMPION_TEST:.3f}**, with an accuracy of **{champion_test_acc:.1%}**. The majority-class baseline reached only **{baseline_f1:.3f}** macro F1.

{('Compared with the untuned version of the same model, the cross-validated macro F1 changed by **' + format(gain, '+.3f') + '**. ') if gain is not None else 'This tuned pipeline has no separately listed untuned counterpart. '}The selected parameters were: {format_params(champion_params)}.

The similar cross-validation and held-out test scores indicate that the selected pipeline generalizes reasonably to unseen data. We now refit this complete preprocessing-and-classification pipeline on the full dataset and save it for the Streamlit application.
"""

display(HTML(f"<div style='font-family:sans-serif; max-width:850px; line-height:1.5;'>{final_decision.replace(chr(10), '<br>')}</div>"))